# Stage 1: Euclid encoder into the DebVAder latent space

Compares the Stage 1 runs (ablations of section 6C) on the D1 test split: training curves, the acceptance gate, metrics binned in magnitude, S/N, size and seeing, and examples. `NB_STAGE1_RUN` picks the run shown in detail.

Paths: **ref** = D_R(mu(E_R(x_R))), **cross** = D_R(mu(E_E(x_E))), **mean** = D_R(mean Rubin latent), the floor.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dp2_cutouts.mlconfig import MLConfig
from dp2_cutouts.ml import figures as F

cfg = MLConfig.load(os.environ['DP2_ML_CONFIG'])
RUN = os.environ.get('NB_STAGE1_RUN', 's1_latent_cross')
IB = list(cfg['rubin']['bands']).index(cfg['rubin']['shape_band'])
runs = [n for n in cfg['stage1']['ablations'] if (cfg.run_dir(n) / 'run.json').exists()]
print('runs found:', runs, '| detail:', RUN)

In [ ]:
F.histories([cfg.run_dir(n) for n in runs], keys=('total_loss', 'latent_loss', 'cross_loss', 'infonce_loss', 'latent_mse'));

## Acceptance gate, all runs

The thresholds (`stage1_acceptance`) are proposals: read them together with the numbers, not instead of them.

In [ ]:
rows = []
for n in runs:
    p = cfg.eval_dir(n) / 'stage1_test_summary.json'
    if not p.exists():
        continue
    s = F.load_json(p)
    r = {'run': n, 'gate': s['gate']['passed']}
    r.update({k: v for k, v in s['summary'].items() if not isinstance(v, (dict, list))})
    r.update({'retrieval_' + k: v for k, v in s['summary']['retrieval'].items()})
    rows.append(r)
table = pd.DataFrame(rows).set_index('run').T
table

In [ ]:
s = F.load_json(cfg.eval_dir(RUN) / 'stage1_test_summary.json')
pd.Series(s['gate']['checks'], name=RUN).to_frame()

## Shapes and fluxes of the Euclid path against the reference path

In [ ]:
df = pd.read_parquet(cfg.eval_dir(RUN) / 'stage1_test.parquet')
F.scatter_e(df, 'cross', 'ref', 'D_R(E_E(x_E))', 'D_R(E_R(x_R))');
F.scatter_e(df, 'mean', 'ref', 'mean latent (floor)', 'D_R(E_R(x_R))');

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, b in zip(ax, ('r', 'i', 'z')):
    a.hist(df['rflux_' + b + '_cross_ref'].clip(-0.5, 0.5), bins=100); a.set_xlabel('flux cross / ref - 1, ' + b)
fig.tight_layout()

In [ ]:
bins = pd.read_parquet(cfg.eval_dir(RUN) / 'stage1_test_bins.parquet')
F.binned_plot(bins, ['mean_de1', 'mean_de2', 'scatter_de1', 'median_rflux_i']);
F.binned_plot(bins, ['median_latent_dist']);

## Examples

Observed Rubin stamp, reference reconstruction, Euclid-path reconstruction (with residuals), and the Euclid input.

In [ ]:
ex = np.load(cfg.eval_dir(RUN) / 'stage1_test_examples.npz')
F.residual_grid(ex['obs'], [ex['ref'], ex['cross']], ['ref', 'cross'], IB, ex['weight'], n=8);
F.stamp_grid([ex['euclid']], ['Euclid input (normalised)'], n=8, size=1.2);

The worst objects for the shape difference, to look at:

In [ ]:
df['de_cross_ref'] = np.hypot(df['de1_cross_ref'], df['de2_cross_ref'])
df.nlargest(15, 'de_cross_ref')[['object_id', 'i_cModel_mag', 'snr_i', 'sigma_hsm', 'ref_e1', 'cross_e1', 'ref_e2', 'cross_e2',
                                 'latent_whitened_dist', 'n_euclid_contaminants', 'astrometric_offset_arcsec']]